In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import recall_score, accuracy_score

attr = pd.read_csv('/kaggle/input/track-7/sevendabang/Antai_AE_round1_item_attr_20190626.csv')       
train = pd.read_csv('/kaggle/input/track-7/sevendabang/Antai_AE_round1_train_20190626.csv')       
test = pd.read_csv('/kaggle/input/track-7/sevendabang/Antai_AE_round1_test_20190626.csv')         
train.drop(columns=['buyer_country_id'],inplace=True)
test.drop(columns=['buyer_country_id'],inplace=True)

print('train',train.head(5))
print('attr',attr.head(5))
print('test',test.head(5))

def calculate_time_decay_co_occurrence(train):
    train['create_order_time'] = pd.to_datetime(train['create_order_time'], errors='coerce')  # 转换为datetime
    # 1. 计算每个用户的购买时间（取最后一次购买该商品的时间）
    user_item_last_time = train.groupby(['buyer_admin_id', 'item_id'])['create_order_time'].max().reset_index()
    # 2. 计算时间衰减因子（距离数据最大时间越近，权重越高）
    max_time = train['create_order_time'].max()
    user_item_last_time['time_decay'] = 1 / (1 + (max_time - user_item_last_time['create_order_time']).dt.days / 30)  # 30天半衰期
    
    # 3. 按用户分组，带时间衰减的物品列表
    user_item_groups = user_item_last_time.groupby('buyer_admin_id').apply(
        lambda x: list(zip(x['item_id'], x['time_decay']))
    ).reset_index()
    
    # 4. 统计带时间衰减的共现矩阵
    C = dict()
    N = dict()
    for _, row in user_item_groups.iterrows():
        items_with_decay = row[0]  # (item_id, time_decay)列表
        items = [i for i, d in items_with_decay]
        decays = [d for i, d in items_with_decay]
        
        # 更新N（用户覆盖度，带衰减）
        for i, d in zip(items, decays):
            if i not in N:
                N[i] = 0
            N[i] += d
        
        # 更新C（共现矩阵，带衰减）
        for i_idx, (i, d_i) in enumerate(items_with_decay):
            if i not in C:
                C[i] = dict()
            for j_idx, (j, d_j) in enumerate(items_with_decay):
                if i == j:
                    continue
                decay = d_i * d_j  # 共现衰减=两个商品的衰减乘积
                if j not in C[i]:
                    C[i][j] = 0
                C[i][j] += decay
    return C, N

def precompute_co_occurrence_recall(C, top_n=20):
    """
    基于共现矩阵，预计算每个商品的Top-N关联召回商品
    C: 物品共现矩阵（C[i][j] = 同时购买i和j的用户数）
    top_n: 每个商品保留的关联商品数量
    返回：dict，key=商品i，value=与i共现的Top-N商品列表（按共现次数降序）
    """
    co_recall_dict = {}
    for i in C.keys():
        # 对商品i的共现商品，按共现次数降序排序，取Top-N
        sorted_co_items = sorted(C[i].items(), key=lambda x: x[1], reverse=True)[:top_n]
        co_recall_dict[i] = [j for j, _ in sorted_co_items]  # 只保留商品ID，不保留次数
    return co_recall_dict

# 1. 先计算共现矩阵（用之前的函数）
C, N = calculate_time_decay_co_occurrence(train)
# 2. 预计算每个商品的Top20关联召回商品
co_recall_dict = precompute_co_occurrence_recall(C, top_n=10)     

train = pd.merge(
    train, 
    attr[['item_id', 'cate_id', 'item_price','store_id']],  # 只关联需要的商品属性
    on='item_id', 
    how='left'  # 左连接，保留train中所有记录
)

test = pd.merge(
    test, 
    attr[['item_id', 'cate_id', 'item_price','store_id']],  # 只关联需要的商品属性
    on='item_id', 
    how='left'  # 左连接，保留train中所有记录
)

def flatten_multiindex(df):
    """
    1. 将 MultiIndex 列名转为单层
    2. 修复关联键列名（如 user_id_ → user_id，merchant_id_ → merchant_id）
    """
    # 步骤1：扁平化 MultiIndex 列名
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ['_'.join(col).strip() for col in df.columns.values]
    
    # 步骤2：修复关联键多余下划线（关键！解决 user_id_ 问题）
    fix_mapping = {
        'buyer_admin_id_': 'buyer_admin_id',
    }
    df = df.rename(columns=lambda x: fix_mapping.get(x, x))  # 只修改带多余下划线的关联键
    
    return df

agg_dict = {
    'item_price':['first','mean','min','max'], 
}
train_user = train.groupby(['buyer_admin_id'], as_index=False).agg(agg_dict)
train_user = flatten_multiindex(train_user)
train = train.merge(train_user, on=['buyer_admin_id'], how='left', suffixes=('', '_user'))

test_user = test.groupby(['buyer_admin_id'], as_index=False).agg(agg_dict)
test_user = flatten_multiindex(test_user)
test = test.merge(test_user, on=['buyer_admin_id'], how='left', suffixes=('', '_user'))

def create_irank1_flag_efficient(df):
    """
    使用合并方法更高效地创建标记
    """
    # 获取每个客户irank=1的item_id
    irank1_items = df[df['irank'] == 1][['buyer_admin_id', 'item_id']].drop_duplicates()
    irank1_items = irank1_items.rename(columns={'item_id': 'irank1_item_id'})
    
    # 合并回原始数据
    df_with_irank1 = df.merge(irank1_items, on='buyer_admin_id', how='left')
    
    # 创建标记列
    df_with_irank1['is_irank1_item'] = (df_with_irank1['item_id'] == df_with_irank1['irank1_item_id']).astype(int)
    
    # 删除临时列
    df_with_irank1 = df_with_irank1.drop('irank1_item_id', axis=1)
    
    return df_with_irank1

# 使用方法2（更高效）创建标记
train = create_irank1_flag_efficient(train)

train_item_ids = set(train['item_id'].unique())
# 对多个irank值进行处理
iranks_to_process = [7] # 可以添加更多irank值，如 [7, 8, 9]
for irank in iranks_to_process:
    mask = test['irank'] == irank
    test.loc[mask, 'item_id'] = np.where(
        test.loc[mask, 'item_id'].isin(train_item_ids),
        test.loc[mask, 'item_id'],        
12691565    
)
test = test[test['item_id'].isin(train_item_ids)].copy()
test = test.sort_values(
    by=['buyer_admin_id', 'irank'],  # 排序的列名列表
    ascending=[True, True]           # 排序方向：用户ID升序，irank升序
).reset_index(drop=True) 
test['irank'] = test.groupby(['buyer_admin_id']).cumcount() + 1

train = train[train['irank']<=50]
test = test[test['irank']<=50]

def transform_date(df):    
# 提取各种时间特征
    df['create_order_time'] = pd.to_datetime(df['create_order_time'])
    df['month'] = df['create_order_time'].dt.month
    df['day'] = df['create_order_time'].dt.day
    df['hour'] = df['create_order_time'].dt.hour
    df['dayofweek'] = df['create_order_time'].dt.dayofweek  # 周一=0, 周日=6
    df['dayofyear'] = df['create_order_time'].dt.dayofyear
    df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)  
    df.drop(columns='create_order_time',inplace=True)    
    return df


def frequency_encoding(df, column, normalize=False):
    """
    频率编码 - 用类别出现的频率代替类别值
    返回编码后的列和频率字典
    """
    freq = df[column].value_counts(normalize=normalize)
    if normalize:
        # 归一化频率 (0-1)
        encoded_col = df[column].map(freq)
    else:
        # 原始计数
        encoded_col = df[column].map(freq)
    
    return encoded_col, freq.to_dict()

# 应用频率编码 - 修复版本
train['item_id_freq'], freq_dict = frequency_encoding(train, 'item_id', normalize=True)
test['item_id_freq'] = test['item_id'].map(freq_dict).fillna(0)  # 使用训练集的频率映射，缺失值填0

# 计算前100个类别的占比
value_counts = train['item_id'].value_counts()
top_100 = value_counts.head(25000)
total_count = value_counts.sum()

# 计算前100个类别的占比
top_100_ratio = top_100.sum() / total_count
top_100_coverage = len(top_100) / len(value_counts) * 100

print(f"总商品种类数: {len(value_counts):,}")
print(f"前100个热门商品购买次数: {top_100.sum():,}")
print(f"总购买次数: {total_count:,}")
print(f"前100个热门商品购买次数占比: {top_100_ratio:.4f} ({top_100_ratio*100:.2f}%)")
print(f"前100个商品种类覆盖率: {top_100_coverage:.4f}%")

len_irank1 = len(train[(train['irank']==1)&(train['is_irank1_item']==1)])/len(train[train['is_irank1_item']==1])
len_irank2 = len(train[(train['irank']==2)&(train['is_irank1_item']==1)])/len(train[train['is_irank1_item']==1])
len_irank3 = len(train[(train['irank']==3)&(train['is_irank1_item']==1)])/len(train[train['is_irank1_item']==1])
len_irank12 = len_irank1 + len_irank2
len_irank123 = len_irank12 + len_irank3

print('irank=1购买label占比:',len_irank1)
print('irank=12购买label占比:',len_irank12)
print('irank=123购买label占比:',len_irank123)


X = train.drop(columns='is_irank1_item')
y = train['is_irank1_item']

X = X[['item_price', 'create_order_time','irank','cate_id','store_id','item_id_freq','item_price_mean','item_price_max','item_price_min','item_price_first']]
X_test = test[['item_price', 'create_order_time','irank','cate_id','store_id','item_id_freq','item_price_mean','item_price_max','item_price_min','item_price_first']]

X = transform_date(X)
X_test = transform_date(X_test)

#################################### cate_id处理 ##########################################
cate_counts = X['cate_id'].value_counts()

# 获取排名前350的cate_id
top_350_cates = cate_counts.head(300).index

# 将排名少于350的cate_id替换为-1
X['cate_id'] = X['cate_id'].apply(
    lambda x: x if x in top_350_cates else -1
)

# 对测试集也做同样的处理
X_test['cate_id'] = X_test['cate_id'].apply(
    lambda x: x if x in top_350_cates else -1
)

X['cate_id'] = X['cate_id'].astype('category')
X_test['cate_id'] = X_test['cate_id'].astype('category')
#################################### store_id处理 ##########################################
store_counts = X['store_id'].value_counts()
# 获取排名前350的cate_id
top_25_store = store_counts.head(25).index
# 将排名少于350的cate_id替换为-1
X['store_id'] = X['store_id'].apply(    
lambda x: x if x in top_25_store else -1
)
# 对测试集也做同样的处理
X_test['store_id'] = X_test['store_id'].apply(    
lambda x: x if x in top_25_store else -1
)
X['store_id'] = X['store_id'].astype('category')
X_test['store_id'] = X_test['store_id'].astype('category')

# X['buyer_country_id'] = X['buyer_country_id'].astype('category')
# X_test['buyer_country_id'] =X_test['buyer_country_id'].astype('category')


lgb_params = {
    'boosting_type': 'gbdt',
    'objective': 'binary',
    'metric': 'binary_logloss',
    'max_depth':6,
    'learning_rate': 0.1,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'seed': 42,
    'verbose':-1,
    'device': 'cpu',
}

fold_recall = []
fold_acc = []
skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
all_preds = []
ens_val = []
ens_test = []

for fold, (train_ix, test_ix) in enumerate(skf.split(X, y)):
    print(f"\n========== Fold {fold+1} ==========")
    
    X_train, X_val = X.iloc[train_ix], X.iloc[test_ix]
    y_train, y_val = y.iloc[train_ix], y.iloc[test_ix]
    

    lgb_clf = lgb.LGBMClassifier(**lgb_params, n_estimators=1000)
    
    # LightGBM使用不同的早停参数
    lgb_clf.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=200),
            lgb.log_evaluation(period=200)        
        ],
        categorical_feature=['cate_id','store_id']
    )

    
    lgb_val_proba = lgb_clf.predict_proba(X_val)[:, 1]
    lgb_test_proba = lgb_clf.predict_proba(X_test)[:, 1]
    lgb_val_proba_ = np.where(lgb_val_proba > 0.5, 1, 0)
    lgb_recall = recall_score(y_val, lgb_val_proba_)
    lgb_acc = accuracy_score(y_val, lgb_val_proba_)
    print('LGB_Recall:',lgb_recall)
    print('LGB_Acc:',lgb_acc)
    fold_recall.append(lgb_recall)
    fold_acc.append(lgb_acc)
    ens_test.append(lgb_test_proba)
mean_recall = np.mean(fold_recall,axis=0)
print('mean_recall:',mean_recall)
mean_acc = np.mean(fold_acc,axis=0)
print('mean_acc:',mean_acc)
mean_preds = np.mean(ens_test,axis=0)
test['label_prob'] = mean_preds
test.to_csv('test_irank1.csv',index=False)

test_irank2 = test[test['irank']>=1]
test_irank2 = test_irank2.sort_values(
    by=['irank','label_prob'],  # 排序的列名列表
    ascending=[True, False]           # 排序方向：用户ID升序，irank升序
).reset_index(drop=True)
test_irank2

test = test.sort_values(
    by=['buyer_admin_id','label_prob'],  # 排序的列名列表
    ascending=[True, False]           # 排序方向：用户ID升序，irank升序
).reset_index(drop=True)

def Repeat_Purchase_Reranking(df):
    df = df.drop_duplicates(subset=['buyer_admin_id', 'item_id'])
    df['irank'] = df.groupby(['buyer_admin_id']).cumcount() + 1
    return df

train = Repeat_Purchase_Reranking(train)
test = Repeat_Purchase_Reranking(test)

test = test[test['irank']<=20]

train = pd.read_csv('/kaggle/input/track-7/sevendabang/Antai_AE_round1_train_20190626.csv') 
train = pd.merge(    
train, 
    attr[['item_id', 'cate_id', 'item_price','store_id']],  # 只关联需要的商品属性
    on='item_id', 
    how='left'  # 左连接，保留train中所有记录
)
train = train[train['buyer_country_id']=='yy']
train.drop(columns=['create_order_time','buyer_country_id'],inplace=True)

# 获取全局热门商品
popular_items = train['item_id'].value_counts().index.tolist()
top_popular_items = popular_items[:30]

# 获取每个cate_id的热门商品（基于训练集）
cate_top_items = {}
for cate_id in test['cate_id'].unique():
    cate_items = train[train['cate_id'] == cate_id]['item_id'].value_counts()
    if len(cate_items) > 0:
        cate_top_items[cate_id] = cate_items.head(15).index.tolist()

# 获取每个客户的倒数第二次和倒数第三次购买记录
last_second_purchases = test[test['irank'] == 1][['buyer_admin_id', 'cate_id']].drop_duplicates()
last_third_purchases = test[test['irank'] == 2][['buyer_admin_id', 'cate_id']].drop_duplicates()

last_second_purchases = last_second_purchases.rename(columns={'cate_id': 'cate1'})
last_third_purchases = last_third_purchases.rename(columns={'cate_id': 'cate2'})

# 创建最终提交DataFrame
final_submission = pd.DataFrame({    
    'buyer_admin_id': test['buyer_admin_id'].unique()
})

# 合并倒数第二次和倒数第三次购买的cate_id信息
final_submission = final_submission.merge(last_second_purchases, on='buyer_admin_id', how='left')
final_submission = final_submission.merge(last_third_purchases, on='buyer_admin_id', how='left')

# 生成预测
for i in range(1, 31):    
    print(i, 'running')
    train_label = test[test['irank'] == int(i)][['buyer_admin_id', 'item_id']]
    train_label = train_label.rename(columns={'item_id': f'predict {i}'})
    final_submission = final_submission.merge(train_label, on='buyer_admin_id', how='left')
    # 填充缺失值
    if final_submission[f'predict {i}'].isna().any():        
        na_mask = final_submission[f'predict {i}'].isna()                              
        for idx in final_submission[na_mask].index:
            # 获取该用户已有的推荐商品
            existing_items = set()       
            hist_item = final_submission.at[idx, 'predict 1']
            for j in range(1, i):
                existing_items.add(final_submission.at[idx, f'predict {j}'])
            
            fill_value = None

            if fill_value is None:
                if hist_item in co_recall_dict:
                    # 从共现列表中找未推荐过的商品
                    for co_item in co_recall_dict[hist_item]:
                        if co_item not in existing_items:
                            fill_value = co_item
                            break
            
            # 先尝试使用cate1的热门商品
            if fill_value is None:
                cate1 = final_submission.at[idx, 'cate1']
                if pd.notna(cate1) and cate1 in cate_top_items and len(cate_top_items[cate1]) > 0:
                    for candidate in cate_top_items[cate1]:
                        if candidate not in existing_items:
                            fill_value = candidate
                            break
            
            # 再尝试使用cate2的热门商品
            if fill_value is None:
                cate2 = final_submission.at[idx, 'cate2']
                if pd.notna(cate2) and cate2 in cate_top_items and len(cate_top_items[cate2]) > 0:
                    for candidate in cate_top_items[cate2]:
                        if candidate not in existing_items:
                            fill_value = candidate
                            break
            
            # 如果没找到合适的cate_id商品，使用全局热门商品
            if fill_value is None:
                for candidate in top_popular_items:
                    if candidate not in existing_items:
                        fill_value = candidate
                        break
            
            # 如果还是没找到，使用默认热门商品
            if fill_value is None:
                fill_value = top_popular_items[(i-1) % len(top_popular_items)]
            
            # 填充缺失值
            final_submission.at[idx, f'predict {i}'] = fill_value

final_submission = final_submission.drop(['cate1', 'cate2'], axis=1)

# 保存结果
final_submission.to_csv('username7.csv', index=False, header=False)